# React — Event handling

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Events have to be clicked to be understood, so this topic leans on the playground:
> `06-events.jsx` (LESSON 22) and `07-event-object.jsx` (LESSON 23-24).

## LESSON 22 — Handling events

Built-in elements take event handlers as props. You already know how they are spelled —
LESSON 9 covered the camelCase rule — and now you get to use them.

**The React API:**

```jsx
function Toolbar() {
  function handleClick() {
    console.log("clicked");
  }

  return <button onClick={handleClick}>Refresh</button>;
}
```

`onClick` is a prop like any other, and its value is a **function**. React keeps it and
calls it when the user clicks. You never call it yourself.

React's naming convention, worth adopting because every codebase uses it: the function is
`handleSomething`, the prop is `onSomething`. You met the same pair in LESSON 16.

### Pass the function. Do not call it.

You have seen this rule before, in LESSON 16, when a *parent* passed a function down to
*your* component. This is the same rule in its other home: React's own event props on
built-in elements.

| correct | wrong |
|---|---|
| `onClick={handleClick}` | `onClick={handleClick()}` |

Braces evaluate immediately (LESSON 8), so `handleClick()` runs **during render** and React
receives whatever it returned — usually `undefined`.

**The symptom is different here, and worth knowing.** In LESSON 16 your child component
called the prop itself, so an `undefined` prop threw
`TypeError: onSelect is not a function`. React's built-in handlers are more forgiving: given
`undefined`, React simply attaches no handler at all. So the button

- runs your function once **while the page is rendering**, before anyone clicks, and
- then does **nothing whatsoever** when clicked. No error, no warning, no clue.

In development you will usually see the early output twice, because `<StrictMode>` renders
components an extra time (LESSON 3). Two log lines before you have touched anything is a
strong hint that a handler is being called instead of passed.

### Passing an argument

`onClick` calls your function for you, so you cannot hand it arguments directly. Wrap it in
an arrow:

```jsx
<button onClick={() => handleGreet("Ada")}>Greet</button>
```

Now the **arrow** is the function React stores. When the click comes, React calls the arrow,
and the arrow calls `handleGreet("Ada")`. Nothing runs during render, because an arrow
function is a value — writing it does not call it.

### What a handler can do today

Log something, work something out, call another function — including a function that arrived
as a prop, which is how a child tells its parent that a click happened (LESSON 16).

What it cannot do yet is **change what is on screen**. That needs state, and state is
topic 9. Everything in this topic is the wiring; topic 9 is what you connect it to.

### Key Notes

- Event props take a **function**: `onClick={handleClick}`.
- `onClick={handleClick()}` calls it during render and hands React the result — the button
  then does nothing, silently.
- Use an arrow to pass arguments: `onClick={() => handleGreet("Ada")}`.
- Convention: the prop is `onX`, the function is `handleX`.

### Example

**Runnable — plain JS.** The difference between passing and calling is a JavaScript fact,
not a React one, and it is worth seeing stripped of JSX. `store()` stands in for React
keeping your handler until the click arrives.

In [ ]:
function l22store(handler) {
  return { handler, type: typeof handler };
}

function l22handleClick() {
  console.log("  handleClick ran");
  return undefined; // like most handlers, it returns nothing
}

console.log("passing the function:");
const l22good = l22store(l22handleClick);
console.log("  stored a", l22good.type);

console.log("calling it by mistake:");
const l22bad = l22store(l22handleClick()); // runs NOW
console.log("  stored a", l22bad.type);

// Later, when the "click" happens:
console.log("click on the good one:");
l22good.handler();
console.log("click on the bad one: nothing to call ->", l22bad.handler);

### Exercise

**Part 1.** Write `l22describe(value)` which returns what React would end up with for an
`onClick` given `value`:

- `"a handler"` when `value` is a function,
- `"nothing to call"` otherwise.

Test it with: a named function, an arrow, the **result** of calling a function, and a string.

**Part 2.** Write `l22makeGreeter(name)` which returns a **function** that logs
`hello, <name>` when called. Store two of them — for `"Ada"` and `"Linus"` — call each one,
and confirm that nothing was logged at the moment you created them.

This is the arrow-wrapping pattern from the lesson, written out: a function that produces a
function, so the work happens later.

**Part 3.** In a comment: why can you not write `onClick={handleGreet("Ada")}` and expect it
to greet Ada on click? Say what React actually receives.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**In the playground.** Point `playground/src/App.jsx` at `./experiments/06-events.jsx` and
run it. Open the console **before** you click anything.

1. Something has already been logged. What, how many times, and why twice? (LESSON 3 has the
   answer to "why twice".)
2. Click each of the three buttons in turn and write down what each one logs.
3. One button does nothing at all. It also produces no error. Explain, in terms of what
   React was handed, why there is nothing to report.
4. Fix the broken button **without** removing the argument-passing behaviour of the middle
   one — that is, keep one button that greets someone by name.

In [ ]:
// Your code here

## LESSON 23 — The event object

Every handler React calls receives one argument: the **React event object** — the docs also
call it a *synthetic event*. It carries what happened, where it happened, and a few methods
for controlling what happens next.

```jsx
function handleClick(event) {
  console.log(event.type);   // "click"
}
```

Four things off it cover almost everything you will need.

### `event.preventDefault()`

> Prevents the default browser action for the event.

Some elements do something on their own: a link navigates, a form submits, a checkbox ticks.
`preventDefault()` cancels that, leaving your handler in charge.

```jsx
function handleLink(event) {
  event.preventDefault();     // the browser does not navigate
  console.log("handled it myself");
}

<a href="https://example.com" onClick={handleLink}>Open</a>
```

### `event.target`

> Returns the node on which the event has occurred (which could be a distant child).

For an input, that node is the input, so `event.target.value` is what the user has typed:

```jsx
<input onChange={(event) => console.log(event.target.value)} />
```

**That is reading, not controlling.** The input above has no `value` prop, so the browser
still owns its contents — React is only listening. Making React own the value is topic 10.

### Bubbling

A click on a button also reaches its ancestors. The handlers run from the inside out —
the element you clicked first, then each parent in turn:

```text
inner button handler
  outer div handler
```

This is usually helpful: it lets one handler high up serve many elements below it, which is
LESSON 24. Occasionally it is not what you want.

### `event.stopPropagation()`

> Stops the event propagation through the React tree.

Call it and the event goes no further up:

```jsx
function handleInner(event) {
  event.stopPropagation();    // the outer handler never runs
  console.log("only me");
}
```

Reach for it when an inner control must not trigger the thing its container does — a delete
button inside a row that is itself clickable, for instance.

> A caveat from React's docs, worth knowing rather than memorising: a few events, such as
> `onAbort` and `onLoad`, do not bubble in the browser but **do** bubble in React.

### Two names that are easy to confuse

| | what it is |
|---|---|
| `event.target` | the node where the event actually happened — possibly a deep child |
| `event.currentTarget` | the node whose handler is running right now |

Click the button inside the div, and in the div's handler `target` is the **button** while
`currentTarget` is the **div**. LESSON 24 puts that difference to work.

### Key Notes

- Handlers receive one argument, the React event object.
- `preventDefault()` cancels the browser's built-in behaviour; `stopPropagation()` stops the
  event reaching ancestors.
- `event.target.value` **reads** an input; it does not make the input controlled.
- Events bubble from the clicked element outwards, innermost handler first.

### Example

**In the playground.** There is no cell to run for this lesson — `preventDefault`, bubbling
and `stopPropagation` are real browser behaviour, and a plain-JavaScript imitation would
teach you my imitation rather than React.

```jsx
function handleLink(event) {
  event.preventDefault();
  console.log("link clicked, navigation prevented");
}

function handleTyping(event) {
  console.log("typed:", event.target.value);
}

function handleInnerStopped(event) {
  event.stopPropagation();
  console.log("inner button handler (propagation stopped)");
}
```

### Exercise

**In the playground.** Point `playground/src/App.jsx` at `./experiments/07-event-object.jsx`
and run it with the console open.

1. Click the link. The console logs, and the page stays where it is. Now comment out the
   `event.preventDefault()` line and click again — what happens, and what does the console
   show before it happens?
2. Put the line back. Type a few characters into the input and watch the log. Then check the
   input has no `value` prop, and say in a comment who currently owns the text in that box.
3. Click **"Click me — both handlers run"**. Two lines appear. Which order are they in, and
   why that order?
4. Click **"Click me — only this one runs"**. Only one line appears. Remove the
   `event.stopPropagation()` call, click again, and confirm the second line comes back.
5. Add `console.log(event.target.id, event.currentTarget.id)` inside the **outer div's**
   handler, then click the inner button. The two ids are different — write down which is
   which.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

Answers in comments.

1. A row in a table is clickable — clicking it opens the record. Inside the row there is a
   small **Delete** button. Clicking Delete also opens the record, which nobody wants. Name
   the method that fixes it and say exactly where it goes.
2. `preventDefault()` and `stopPropagation()` are often typed together out of habit.
   Describe a case where you want one and definitely not the other.
3. A colleague writes `<input value={event.target.value} />` after reading this lesson.
   Explain why that cannot work as written — and which topic actually covers what they are
   reaching for.

## LESSON 24 — One handler for many controls

A toolbar with three buttons does not need three handlers:

```jsx
<button onClick={handleRefresh}>Refresh</button>
<button onClick={handleExport}>Export</button>
<button onClick={handleArchive}>Archive</button>
```

Three functions that differ by one word each, and a fourth button means a fourth function.
When the controls are variations on one job, one handler does better — but it needs to know
**which control was used**.

### Let the element carry its own identity

```jsx
function handleToolbar(event) {
  console.log("action:", event.currentTarget.dataset.action);
}

<button data-action="refresh" onClick={handleToolbar}>Refresh</button>
<button data-action="export"  onClick={handleToolbar}>Export</button>
<button data-action="archive" onClick={handleToolbar}>Archive</button>
```

`data-*` attributes are plain HTML, and the browser collects them into `dataset`. One
handler, and each button announces itself.

This is also why the list version is so short — the buttons can come straight from data
(LESSON 19, with a key from LESSON 20):

```jsx
{actions.map((action) => (
  <button key={action.id} data-action={action.id} onClick={handleToolbar}>
    {action.label}
  </button>
))}
```

### Why `currentTarget` and not `target`

LESSON 23 gave the definitions: `target` is where the event happened, `currentTarget` is the
element whose handler is running. Here is why that matters in practice.

Put anything inside the button — an icon, a `<span>`, bold text — and a click can land on
that child. Measured, from the playground experiment:

```text
clicking the button        -> action: refresh | target was: BUTTON
clicking the span inside   -> action: refresh | target was: SPAN
```

`event.target.dataset.action` would be `undefined` in the second case: the span has no
`data-action`. `currentTarget` is the button both times, because the button is where the
handler is attached.

**Use `currentTarget` when you want the element you wired up.** Reach for `target` only when
you genuinely care where the click landed.

### Dispatching

Once you have the action name, a lookup object turns it into behaviour — the same shape as
LESSON 13's class lookup:

```js
const actions = {
  refresh: () => console.log("refreshing"),
  export: () => console.log("exporting"),
};

actions[name]?.();
```

The `?.()` matters: an unknown name gives `undefined`, and calling `undefined` would throw.
This way an unrecognised action does nothing instead of breaking the page.

### When one handler is the wrong choice

Reuse is worth it when the controls really are variations of one job. If the three buttons
share nothing but their container — one opens a dialog, one submits a request, one toggles a
panel — then a single handler becomes a switchboard that has to know about all three, and
three small functions are clearer.

The test: **does the handler read as one job, or as three jobs behind a lookup?**

### Key Notes

- One handler plus `data-*` beats one handler per control when the controls are variations of
  one job.
- Read the identity off `event.currentTarget` — `target` may be a child element.
- A lookup object turns an action name into behaviour; `?.()` keeps unknown names harmless.
- Do not merge handlers that have nothing in common.

### Example

**Runnable — plain JS.** The dispatch half is ordinary JavaScript, and it is the half you
will reuse everywhere. Only the reading of `currentTarget` needs a browser.

In [ ]:
const l24actions = {
  refresh: () => "refreshing the list",
  export: () => "exporting to CSV",
  archive: () => "archiving selection",
};

function l24dispatch(name) {
  const action = l24actions[name];
  return action?.() ?? `no action called "${name}"`;
}

console.log(l24dispatch("refresh"));
console.log(l24dispatch("export"));
console.log(l24dispatch("delete")); // not in the table - handled, not thrown

### Exercise

**Part 1.** A row of filter buttons carries `data-filter` values `"all"`, `"unpaid"` and
`"overdue"`. Write `l24describeFilter(name)` which returns:

- `"showing everything"` for `all`
- `"showing unpaid invoices"` for `unpaid`
- `"showing overdue invoices"` for `overdue`
- `"unknown filter: <name>"` for anything else — **without** throwing.

Use a lookup object, not a chain of `if`s. Test all four cases.

**Part 2.** Write `l24readAction(elementLike)` which takes an object shaped like an event —
`{ currentTarget: { dataset: { action } }, target: { dataset: {} } }` — and returns the
action from **`currentTarget`**. Then call it with an object where `target.dataset.action` is
missing but `currentTarget.dataset.action` is `"export"`, and show it still works.

This is the `target`/`currentTarget` difference written as data, so you can see which one the
code depends on.

**Part 3.** In a comment: you add an icon `<span>` inside each button. Which of the two
properties keeps working, and which breaks?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1 — in the playground.** `./experiments/07-event-object.jsx` has a three-button
toolbar at the bottom, all wired to one handler.

1. Click each button and note what is logged.
2. Click directly on the **text inside** a button (the `<span>`). Compare the `target was:`
   part of the log with the previous clicks. The action is still correct — say why.
3. Change `event.currentTarget` to `event.target` in the handler, then click the span again.
   What does the action become, and why?
4. Put it back.

**Part 2 — judgement.** Answer in comments.

1. Five buttons: `Bold`, `Italic`, `Underline`, `Bullet list`, `Numbered list`. One handler
   or five? Justify it in one sentence.
2. Three buttons: `Save draft`, `Open settings dialog`, `Log out`. One handler or three?
3. What would you lose by giving every button `data-action` and routing all of them through a
   single handler for the whole application?